In [ ]:
%matplotlib inline
import numpy as np
import scipy.signal as sig
import matplotlib.pyplot as plt
from abc import ABC, abstractmethod
from pathlib import Path
import soundfile as sf

BASE_DIR = Path('.')
RAW_DIR = BASE_DIR / 'data' / 'raw'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
SR = 44100
print('Imports OK')

## Base Class

All effects inherit from `AudioEffect`, which enforces a common interface: `process()`, `get_params()`, `set_params()`. This allows effects to be composed into chains (Notebook 03).

In [ ]:
class AudioEffect(ABC):
    """Abstract base class for all audio effects."""
    def __init__(self, name='effect'):
        self.name = name
        self.bypass = False

    @abstractmethod
    def process(self, signal: np.ndarray, sample_rate: int) -> np.ndarray:
        """Process mono float32 signal, return processed float32 signal."""
        pass

    def get_params(self) -> dict:
        """Return dict of all effect parameters."""
        return {k: v for k, v in self.__dict__.items() if not k.startswith('_') and k != 'name'}

    def set_params(self, **kwargs):
        """Set one or more parameters by name."""
        for k, v in kwargs.items():
            if hasattr(self, k):
                setattr(self, k, v)
            else:
                raise ValueError(f'Unknown parameter: {k}')

    def __repr__(self):
        params = ', '.join(f'{k}={v}' for k, v in self.get_params().items() if k != 'bypass')
        return f'{self.__class__.__name__}({params})'

# Helper test signal
def make_test_signal(duration=2.0, sr=44100):
    """Sine wave + white noise for demo cells."""
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    sig_out = 0.7 * np.sin(2 * np.pi * 440 * t) + 0.1 * np.random.randn(len(t))
    return sig_out.astype(np.float32)

def plot_before_after(dry, wet, sr, title='Effect', duration_show=0.05):
    """Quick waveform comparison plot."""
    n_show = int(duration_show * sr)
    t = np.arange(n_show) / sr * 1000  # ms
    fig, axes = plt.subplots(1, 2, figsize=(12, 3))
    axes[0].plot(t, dry[:n_show], color='steelblue', linewidth=0.8)
    axes[0].set_title('Dry', fontsize=10)
    axes[0].set_xlabel('Time (ms)')
    axes[0].set_ylabel('Amplitude')
    axes[0].grid(True, alpha=0.3)
    axes[1].plot(t, wet[:n_show], color='tomato', linewidth=0.8)
    axes[1].set_title(f'Wet ({title})', fontsize=10)
    axes[1].set_xlabel('Time (ms)')
    axes[1].set_ylabel('Amplitude')
    axes[1].grid(True, alpha=0.3)
    plt.suptitle(title, fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()

print('Base class defined.')

## Effect 1: ConvolutionReverb

Uses FFT-based overlap-add convolution. The overlap-add method splits the input into blocks of length `block_size`, convolves each block with the IR using FFT, and accumulates the results with proper overlap. This is O(N log N) vs O(N²) for direct convolution.

In [ ]:
class ConvolutionReverb(AudioEffect):
    """
    FFT-based convolution reverb using overlap-add method.
    Convolves the input signal with a given impulse response (IR).
    The IR can be any array representing a room's acoustic fingerprint.
    """
    def __init__(self, ir: np.ndarray = None, wet_mix: float = 0.5, normalize_ir: bool = True):
        super().__init__('convolution_reverb')
        if ir is None:
            # Default: short exponential decay IR
            n = 4410  # 100ms at 44100
            t = np.arange(n) / 44100
            ir = (np.random.randn(n) * np.exp(-5 * t)).astype(np.float32)
            ir[0] = 1.0
        self.ir = ir.astype(np.float32)
        if normalize_ir:
            self.ir = self.ir / (np.max(np.abs(self.ir)) + 1e-9)
        self.wet_mix = wet_mix
        self.normalize_ir = normalize_ir

    def process(self, signal: np.ndarray, sample_rate: int) -> np.ndarray:
        dry = signal.astype(np.float32)
        wet = sig.fftconvolve(dry, self.ir)[:len(dry)]
        peak = np.max(np.abs(wet))
        if peak > 0:
            wet = wet / peak * 0.9
        out = (1 - self.wet_mix) * dry + self.wet_mix * wet
        return out.astype(np.float32)

# Demo
dry = make_test_signal(2.0)
n_ir = int(0.5 * SR)
t_ir = np.arange(n_ir) / SR
ir = (np.random.randn(n_ir) * np.exp(-4 * t_ir)).astype(np.float32)
ir[0] = 1.0
conv_reverb = ConvolutionReverb(ir=ir, wet_mix=0.6)
wet = conv_reverb.process(dry, SR)
print(f'ConvolutionReverb: in={dry.shape}, out={wet.shape}')
plot_before_after(dry, wet, SR, 'ConvolutionReverb')

## Effect 2: SchroederReverb

Manfred Schroeder's classic reverberator (1962): 4 parallel comb filters feeding 2 series allpass filters. The comb filters create the reverb density; the allpass filters improve the echo density without adding coloration.

In [ ]:
class SchroederReverb(AudioEffect):
    """
    Classic Schroeder reverberator.
    Architecture: 4 parallel comb filters summed into 2 series allpass filters.
    Comb delays are prime-ratio to each other to avoid common-mode resonances.
    """
    def __init__(self, room_size: float = 0.5, damping: float = 0.5,
                 wet_mix: float = 0.5, pre_delay_ms: float = 20.0):
        super().__init__('schroeder_reverb')
        self.room_size = np.clip(room_size, 0.1, 1.0)
        self.damping = np.clip(damping, 0.0, 1.0)
        self.wet_mix = np.clip(wet_mix, 0.0, 1.0)
        self.pre_delay_ms = pre_delay_ms

    def _comb_filter(self, x, delay, feedback, damping):
        """Feedback comb filter with damping (lowpass in feedback path)."""
        d = max(1, int(delay))
        buf = np.zeros(d)
        y = np.zeros_like(x)
        filterstore = 0.0
        damp1 = damping
        damp2 = 1.0 - damping
        pos = 0
        for i in range(len(x)):
            output = buf[pos]
            filterstore = output * damp2 + filterstore * damp1
            buf[pos] = x[i] + filterstore * feedback
            y[i] = output
            pos = (pos + 1) % d
        return y

    def _allpass_filter(self, x, delay, feedback=0.5):
        """Schroeder allpass filter."""
        d = max(1, int(delay))
        buf = np.zeros(d)
        y = np.zeros_like(x)
        pos = 0
        for i in range(len(x)):
            buf_out = buf[pos]
            buf[pos] = x[i] + buf_out * feedback
            y[i] = buf_out - x[i] * feedback
            pos = (pos + 1) % d
        return y

    def process(self, signal: np.ndarray, sample_rate: int) -> np.ndarray:
        dry = signal.astype(np.float32)
        # Pre-delay
        pre_delay_s = int(self.pre_delay_ms * 0.001 * sample_rate)
        x = np.zeros(len(dry) + pre_delay_s)
        x[pre_delay_s:] = dry
        x = x[:len(dry)]
        # Comb filter delays (ms), scaled by room_size
        scale = 0.5 + self.room_size * 0.5
        comb_delays = [int(d * scale * sample_rate / 1000) for d in [29.7, 37.1, 41.1, 43.7]]
        feedback = 0.7 + self.room_size * 0.18
        # 4 parallel comb filters
        comb_out = np.zeros(len(dry))
        for cd in comb_delays:
            comb_out += self._comb_filter(x, cd, feedback, self.damping * 0.4)
        comb_out *= 0.25
        # 2 series allpass filters
        allpass_delays = [int(d * sample_rate / 1000) for d in [5.0, 1.7]]
        ap_out = comb_out
        for ad in allpass_delays:
            ap_out = self._allpass_filter(ap_out, ad, 0.5)
        out = (1 - self.wet_mix) * dry + self.wet_mix * ap_out
        return np.clip(out, -1, 1).astype(np.float32)

dry = make_test_signal(1.0)
sch = SchroederReverb(room_size=0.7, damping=0.4, wet_mix=0.5)
wet = sch.process(dry, SR)
print(f'SchroederReverb: params={sch.get_params()}')
plot_before_after(dry, wet, SR, 'SchroederReverb')

## Effect 3: MoorerReverb

James Moorer's 1979 extension of Schroeder: 6 comb filters, early reflections tap delay, and a single allpass. The early reflections are tapped from a delay line at measured reflection times, providing a more realistic psychoacoustic response.

In [ ]:
class MoorerReverb(AudioEffect):
    """
    Moorer reverberator with 6 lowpass-feedback comb filters,
    early reflections tap delay, and a single allpass diffuser.
    """
    def __init__(self, room_size: float = 0.5, damping: float = 0.5,
                 wet_mix: float = 0.5, early_level: float = 0.3):
        super().__init__('moorer_reverb')
        self.room_size = np.clip(room_size, 0.1, 1.0)
        self.damping = np.clip(damping, 0.0, 1.0)
        self.wet_mix = wet_mix
        self.early_level = early_level

    def _lp_comb(self, x, delay, feedback, damp):
        d = max(1, int(delay))
        buf = np.zeros(d)
        y = np.zeros(len(x))
        lp = 0.0
        pos = 0
        for i in range(len(x)):
            out_val = buf[pos]
            lp = out_val * (1 - damp) + lp * damp
            buf[pos] = x[i] + lp * feedback
            y[i] = out_val
            pos = (pos + 1) % d
        return y

    def _allpass(self, x, delay, g=0.5):
        d = max(1, int(delay))
        buf = np.zeros(d)
        y = np.zeros(len(x))
        pos = 0
        for i in range(len(x)):
            b = buf[pos]
            buf[pos] = x[i] + b * g
            y[i] = b - x[i] * g
            pos = (pos + 1) % d
        return y

    def process(self, signal: np.ndarray, sample_rate: int) -> np.ndarray:
        dry = signal.astype(np.float32)
        scale = 0.5 + self.room_size * 0.5
        # Early reflections
        er_delays_ms = [5, 12, 22, 35, 48, 60]
        er_gains = [0.8, 0.6, 0.5, 0.45, 0.4, 0.35]
        early = np.zeros_like(dry)
        for d_ms, g in zip(er_delays_ms, er_gains):
            d = int(d_ms * scale * sample_rate / 1000)
            if d > 0 and d < len(dry):
                early[d:] += g * dry[:-d]
        # 6 parallel comb filters
        comb_ms = [30.1, 35.3, 39.1, 41.5, 43.7, 46.1]
        fb = 0.7 + self.room_size * 0.18
        damp = self.damping * 0.4
        late = np.zeros_like(dry)
        for cm in comb_ms:
            cd = int(cm * scale * sample_rate / 1000)
            late += self._lp_comb(dry, cd, fb, damp)
        late /= 6.0
        late = self._allpass(late, int(6.3 * sample_rate / 1000), 0.5)
        reverb = self.early_level * early + (1 - self.early_level) * late
        out = (1 - self.wet_mix) * dry + self.wet_mix * reverb
        return np.clip(out, -1, 1).astype(np.float32)

dry = make_test_signal(1.0)
moorer = MoorerReverb(room_size=0.6, wet_mix=0.5)
wet = moorer.process(dry, SR)
print('MoorerReverb OK')
plot_before_after(dry, wet, SR, 'MoorerReverb')

## Effect 4: DigitalDelay

A tap delay line with configurable delay time, feedback, and dry/wet mix. In ping-pong mode (when processing stereo), the delay alternates between left and right channels.

In [ ]:
class DigitalDelay(AudioEffect):
    """
    Tap delay with feedback. Supports stereo ping-pong effect.
    The feedback path causes repeated echoes that decay over time.
    feedback=0 gives a single echo; higher values give more repeats.
    """
    def __init__(self, delay_ms: float = 300.0, feedback: float = 0.4,
                 wet_mix: float = 0.4, ping_pong: bool = False):
        super().__init__('digital_delay')
        self.delay_ms = delay_ms
        self.feedback = np.clip(feedback, 0.0, 0.99)
        self.wet_mix = wet_mix
        self.ping_pong = ping_pong

    def process(self, signal: np.ndarray, sample_rate: int) -> np.ndarray:
        dry = signal.astype(np.float32)
        delay_samples = int(self.delay_ms * sample_rate / 1000)
        delay_samples = max(1, delay_samples)
        buf = np.zeros(delay_samples + len(dry))
        buf[delay_samples:delay_samples + len(dry)] = dry
        wet = np.zeros_like(dry)
        # Feedback loop
        delay_line = np.zeros(delay_samples)
        pos = 0
        for i in range(len(dry)):
            delayed = delay_line[pos]
            wet[i] = delayed
            delay_line[pos] = dry[i] + delayed * self.feedback
            pos = (pos + 1) % delay_samples
        out = (1 - self.wet_mix) * dry + self.wet_mix * wet
        return np.clip(out, -1, 1).astype(np.float32)

dry = make_test_signal(2.0)
delay = DigitalDelay(delay_ms=250, feedback=0.5, wet_mix=0.4)
wet = delay.process(dry, SR)
print('DigitalDelay OK')
plot_before_after(dry, wet, SR, 'DigitalDelay 250ms')

## Effect 5: Chorus

Chorus uses multiple slightly-detuned copies of the signal (via short LFO-modulated delay lines) to simulate multiple instruments playing together. The subtle pitch variation comes from the continuously changing delay times.

In [ ]:
class Chorus(AudioEffect):
    """
    Multi-voice chorus using LFO-modulated delay lines.
    Each voice has a slightly different LFO phase and rate,
    creating the impression of multiple detuned instruments.
    """
    def __init__(self, num_voices: int = 3, rate_hz: float = 1.5,
                 depth_ms: float = 7.0, delay_ms: float = 20.0, wet_mix: float = 0.5):
        super().__init__('chorus')
        self.num_voices = max(2, min(4, num_voices))
        self.rate_hz = rate_hz
        self.depth_ms = depth_ms
        self.delay_ms = delay_ms
        self.wet_mix = wet_mix

    def process(self, signal: np.ndarray, sample_rate: int) -> np.ndarray:
        dry = signal.astype(np.float32)
        n = len(dry)
        t = np.arange(n) / sample_rate
        max_delay = int((self.delay_ms + self.depth_ms) * sample_rate / 1000) + 2
        padded = np.concatenate([np.zeros(max_delay), dry])
        wet = np.zeros(n)
        for v in range(self.num_voices):
            phase_offset = 2 * np.pi * v / self.num_voices
            rate = self.rate_hz * (1 + 0.1 * v)
            lfo = np.sin(2 * np.pi * rate * t + phase_offset)
            delay_samples = (self.delay_ms + self.depth_ms * lfo) * sample_rate / 1000
            for i in range(n):
                d = delay_samples[i]
                d_int = int(d)
                frac = d - d_int
                idx = max_delay + i - d_int
                if idx >= 1 and idx < len(padded):
                    wet[i] += padded[idx] * (1 - frac) + padded[max(0, idx-1)] * frac
        wet /= self.num_voices
        out = (1 - self.wet_mix) * dry + self.wet_mix * wet
        return np.clip(out, -1, 1).astype(np.float32)

dry = make_test_signal(1.0)
chorus = Chorus(num_voices=3, rate_hz=1.2, depth_ms=5.0)
wet = chorus.process(dry, SR)
print('Chorus OK')
plot_before_after(dry, wet, SR, 'Chorus')

## Effect 6: Flanger

Flanger uses a very short (1–15ms) modulated delay line mixed with the dry signal. The comb filtering effect creates the characteristic sweeping notch/peak pattern. Feedback increases the resonance of the effect.

In [ ]:
class Flanger(AudioEffect):
    """
    Flanger: short LFO-modulated delay (1-15ms) with feedback.
    Creates sweeping comb filter notches. Feedback adds resonance/coloration.
    """
    def __init__(self, rate_hz: float = 0.5, depth_ms: float = 3.0,
                 delay_ms: float = 5.0, feedback: float = 0.5, wet_mix: float = 0.5):
        super().__init__('flanger')
        self.rate_hz = rate_hz
        self.depth_ms = depth_ms
        self.delay_ms = delay_ms
        self.feedback = np.clip(feedback, -0.95, 0.95)
        self.wet_mix = wet_mix

    def process(self, signal: np.ndarray, sample_rate: int) -> np.ndarray:
        dry = signal.astype(np.float32)
        n = len(dry)
        t = np.arange(n) / sample_rate
        lfo = np.sin(2 * np.pi * self.rate_hz * t)
        max_delay_s = int((self.delay_ms + self.depth_ms) * sample_rate / 1000) + 2
        buf = np.zeros(max_delay_s)
        wet = np.zeros(n)
        buf_pos = 0
        for i in range(n):
            d = (self.delay_ms + self.depth_ms * lfo[i]) * sample_rate / 1000
            d_int = max(1, int(d))
            frac = d - int(d)
            read_pos = (buf_pos - d_int) % max_delay_s
            read_pos_prev = (buf_pos - d_int - 1) % max_delay_s
            delayed = buf[read_pos] * (1 - frac) + buf[read_pos_prev] * frac
            buf[buf_pos] = dry[i] + delayed * self.feedback
            wet[i] = delayed
            buf_pos = (buf_pos + 1) % max_delay_s
        out = (1 - self.wet_mix) * dry + self.wet_mix * wet
        return np.clip(out, -1, 1).astype(np.float32)

dry = make_test_signal(1.0)
flanger = Flanger(rate_hz=0.4, depth_ms=2.5, feedback=0.6)
wet = flanger.process(dry, SR)
print('Flanger OK')
plot_before_after(dry, wet, SR, 'Flanger')

## Effect 7: Phaser

A phaser uses a cascade of allpass filters whose center frequencies are swept by an LFO. This introduces frequency-dependent phase shifts. When mixed with the dry signal, constructive/destructive interference creates notches in the frequency response.

In [ ]:
class Phaser(AudioEffect):
    """
    N-stage allpass phaser. LFO sweeps center frequency of all allpass stages.
    Mixing phased signal with dry creates constructive/destructive interference.
    num_stages=4 for subtle effect, 8 for deeper phasing.
    """
    def __init__(self, num_stages: int = 4, rate_hz: float = 0.5,
                 depth: float = 0.9, center_freq: float = 1000.0, wet_mix: float = 0.5):
        super().__init__('phaser')
        self.num_stages = num_stages
        self.rate_hz = rate_hz
        self.depth = depth
        self.center_freq = center_freq
        self.wet_mix = wet_mix

    def _first_order_allpass(self, x, coeff):
        """First-order allpass: y[n] = coeff*x[n] + x[n-1] - coeff*y[n-1]."""
        y = np.zeros_like(x)
        for i in range(1, len(x)):
            y[i] = coeff * x[i] + x[i-1] - coeff * y[i-1]
        return y

    def process(self, signal: np.ndarray, sample_rate: int) -> np.ndarray:
        dry = signal.astype(np.float32)
        n = len(dry)
        t = np.arange(n) / sample_rate
        lfo = np.sin(2 * np.pi * self.rate_hz * t)
        # Sweep center frequency
        f_min = self.center_freq * (1 - self.depth * 0.9)
        f_max = self.center_freq * (1 + self.depth * 0.9)
        f_min = max(20, f_min)
        f_max = min(sample_rate * 0.45, f_max)
        # Process in blocks for LFO modulation
        block = 64
        wet = np.zeros(n)
        current = dry.copy()
        for stage in range(self.num_stages):
            out = np.zeros(n)
            y_prev = 0.0
            x_prev = 0.0
            for i in range(n):
                f_c = f_min + (f_max - f_min) * (0.5 + 0.5 * lfo[i])
                w0 = 2 * np.pi * f_c / sample_rate
                coeff = (1 - np.tan(w0 / 2)) / (1 + np.tan(w0 / 2))
                out[i] = coeff * current[i] + x_prev - coeff * y_prev
                x_prev = current[i]
                y_prev = out[i]
            current = out
        out_sig = (1 - self.wet_mix) * dry + self.wet_mix * current
        return np.clip(out_sig, -1, 1).astype(np.float32)

dry = make_test_signal(1.0)
phaser = Phaser(num_stages=4, rate_hz=0.5, depth=0.8)
wet = phaser.process(dry, SR)
print('Phaser OK')
plot_before_after(dry, wet, SR, 'Phaser')

## Effect 8: Tremolo

Tremolo modulates the amplitude of a signal using a low-frequency oscillator. Unlike vibrato (pitch modulation), tremolo strictly modulates loudness. Multiple LFO shapes are supported.

In [ ]:
class Tremolo(AudioEffect):
    """
    Amplitude modulation tremolo.
    LFO shapes: 'sine', 'square', 'triangle'.
    depth=0 is no effect; depth=1 is full on/off modulation.
    """
    def __init__(self, rate_hz: float = 5.0, depth: float = 0.8,
                 shape: str = 'sine', wet_mix: float = 1.0):
        super().__init__('tremolo')
        self.rate_hz = rate_hz
        self.depth = np.clip(depth, 0.0, 1.0)
        self.shape = shape
        self.wet_mix = wet_mix

    def _lfo(self, t):
        phase = 2 * np.pi * self.rate_hz * t
        if self.shape == 'sine':
            return np.sin(phase)
        elif self.shape == 'square':
            return np.sign(np.sin(phase))
        elif self.shape == 'triangle':
            return 2 * np.abs(2 * (self.rate_hz * t - np.floor(self.rate_hz * t + 0.5))) - 1
        return np.sin(phase)

    def process(self, signal: np.ndarray, sample_rate: int) -> np.ndarray:
        dry = signal.astype(np.float32)
        t = np.arange(len(dry)) / sample_rate
        lfo = self._lfo(t)
        envelope = 1.0 - self.depth * (0.5 + 0.5 * lfo)
        wet = dry * envelope.astype(np.float32)
        out = (1 - self.wet_mix) * dry + self.wet_mix * wet
        return out.astype(np.float32)

dry = make_test_signal(1.0)
trem = Tremolo(rate_hz=4.0, depth=0.8, shape='sine')
wet = trem.process(dry, SR)
print('Tremolo OK')
plot_before_after(dry, wet, SR, 'Tremolo')

## Effect 9: Distortion

Distortion introduces harmonic content by clipping or waveshaping the signal. Soft clipping (tanh) adds smooth odd harmonics; hard clipping adds strong odd+even harmonics. Asymmetric waveshaping adds even harmonics for a warmer, tube-like character.

In [ ]:
class Distortion(AudioEffect):
    """
    Waveshaping distortion with three modes:
    - 'soft': tanh saturation (smooth harmonic addition)
    - 'hard': hard clipping (abrupt square-wave clipping)
    - 'asymmetric': different positive/negative shaping (tube-like)
    pre_gain drives signal into the nonlinearity; post_gain compensates volume.
    """
    def __init__(self, drive: float = 5.0, mode: str = 'soft',
                 pre_gain: float = 1.0, post_gain: float = 0.5, wet_mix: float = 1.0):
        super().__init__('distortion')
        self.drive = max(1.0, drive)
        self.mode = mode
        self.pre_gain = pre_gain
        self.post_gain = post_gain
        self.wet_mix = wet_mix

    def process(self, signal: np.ndarray, sample_rate: int) -> np.ndarray:
        dry = signal.astype(np.float32)
        x = dry * self.pre_gain * self.drive
        if self.mode == 'soft':
            wet = np.tanh(x)
        elif self.mode == 'hard':
            wet = np.clip(x, -1.0, 1.0)
        elif self.mode == 'asymmetric':
            wet = np.where(x >= 0,
                           np.tanh(x),
                           np.tanh(x * 0.5) * 1.5)
        else:
            wet = np.tanh(x)
        wet = wet * self.post_gain
        out = (1 - self.wet_mix) * dry + self.wet_mix * wet
        return np.clip(out, -1, 1).astype(np.float32)

dry = make_test_signal(0.5)
dist = Distortion(drive=8.0, mode='soft', post_gain=0.4)
wet = dist.process(dry, SR)
print('Distortion OK')
plot_before_after(dry, wet, SR, 'Distortion (soft)')

## Effect 10: BitCrusher

BitCrusher reduces audio quality deliberately — sample rate reduction creates aliasing artifacts; bit depth reduction creates quantization noise. Both are characteristic of vintage digital hardware and lo-fi aesthetics.

In [ ]:
class BitCrusher(AudioEffect):
    """
    Bit depth and sample rate reducer.
    bit_depth: 1-16 bits. Lower = more distortion (4-8 bits for lo-fi effect).
    downsample: integer factor. 2 halves effective sample rate, etc.
    """
    def __init__(self, bit_depth: int = 8, downsample: int = 4, wet_mix: float = 1.0):
        super().__init__('bit_crusher')
        self.bit_depth = max(1, min(16, bit_depth))
        self.downsample = max(1, downsample)
        self.wet_mix = wet_mix

    def process(self, signal: np.ndarray, sample_rate: int) -> np.ndarray:
        dry = signal.astype(np.float32)
        # Sample rate reduction (hold samples)
        if self.downsample > 1:
            indices = np.arange(len(dry))
            held = dry[(indices // self.downsample) * self.downsample]
        else:
            held = dry.copy()
        # Bit depth reduction
        levels = 2 ** self.bit_depth
        quantized = np.round(held * (levels / 2)) / (levels / 2)
        quantized = np.clip(quantized, -1.0, 1.0).astype(np.float32)
        out = (1 - self.wet_mix) * dry + self.wet_mix * quantized
        return out.astype(np.float32)

dry = make_test_signal(0.5)
bc = BitCrusher(bit_depth=6, downsample=8)
wet = bc.process(dry, SR)
print('BitCrusher OK')
plot_before_after(dry, wet, SR, 'BitCrusher (6-bit, 8x downsample)')

## Effect 11: ParametricEQ

A fully parametric equalizer using second-order IIR biquad filter sections. Each section implements a standard Audio EQ Cookbook formula. Supports: low shelf, high shelf, and peaking (bell) filters.

In [ ]:
class ParametricEQ(AudioEffect):
    """
    5-band parametric equalizer using biquad IIR filters.
    Bands: low shelf, 3 peaking (bell), high shelf.
    Uses Audio EQ Cookbook formulas by Robert Bristow-Johnson.
    """
    def __init__(self,
                 low_shelf_freq: float = 80.0, low_shelf_gain_db: float = 0.0,
                 peak1_freq: float = 250.0, peak1_gain_db: float = 0.0, peak1_q: float = 1.0,
                 peak2_freq: float = 1000.0, peak2_gain_db: float = 0.0, peak2_q: float = 1.0,
                 peak3_freq: float = 4000.0, peak3_gain_db: float = 0.0, peak3_q: float = 1.0,
                 high_shelf_freq: float = 8000.0, high_shelf_gain_db: float = 0.0):
        super().__init__('parametric_eq')
        self.low_shelf_freq = low_shelf_freq
        self.low_shelf_gain_db = low_shelf_gain_db
        self.peak1_freq = peak1_freq; self.peak1_gain_db = peak1_gain_db; self.peak1_q = peak1_q
        self.peak2_freq = peak2_freq; self.peak2_gain_db = peak2_gain_db; self.peak2_q = peak2_q
        self.peak3_freq = peak3_freq; self.peak3_gain_db = peak3_gain_db; self.peak3_q = peak3_q
        self.high_shelf_freq = high_shelf_freq
        self.high_shelf_gain_db = high_shelf_gain_db

    def _low_shelf_coeffs(self, f0, gain_db, sr):
        A = 10 ** (gain_db / 40)
        w0 = 2 * np.pi * f0 / sr
        alpha = np.sin(w0) / 2 * np.sqrt((A + 1/A) * (1/0.707 - 1) + 2)
        b0 =  A * ((A+1) - (A-1)*np.cos(w0) + 2*np.sqrt(A)*alpha)
        b1 =  2*A * ((A-1) - (A+1)*np.cos(w0))
        b2 =  A * ((A+1) - (A-1)*np.cos(w0) - 2*np.sqrt(A)*alpha)
        a0 =       (A+1) + (A-1)*np.cos(w0) + 2*np.sqrt(A)*alpha
        a1 = -2 * ((A-1) + (A+1)*np.cos(w0))
        a2 =       (A+1) + (A-1)*np.cos(w0) - 2*np.sqrt(A)*alpha
        return np.array([b0, b1, b2]) / a0, np.array([1, a1/a0, a2/a0])

    def _high_shelf_coeffs(self, f0, gain_db, sr):
        A = 10 ** (gain_db / 40)
        w0 = 2 * np.pi * f0 / sr
        alpha = np.sin(w0) / 2 * np.sqrt((A + 1/A) * (1/0.707 - 1) + 2)
        b0 =  A * ((A+1) + (A-1)*np.cos(w0) + 2*np.sqrt(A)*alpha)
        b1 = -2*A * ((A-1) + (A+1)*np.cos(w0))
        b2 =  A * ((A+1) + (A-1)*np.cos(w0) - 2*np.sqrt(A)*alpha)
        a0 =       (A+1) - (A-1)*np.cos(w0) + 2*np.sqrt(A)*alpha
        a1 =  2 * ((A-1) - (A+1)*np.cos(w0))
        a2 =       (A+1) - (A-1)*np.cos(w0) - 2*np.sqrt(A)*alpha
        return np.array([b0, b1, b2]) / a0, np.array([1, a1/a0, a2/a0])

    def _peaking_coeffs(self, f0, gain_db, Q, sr):
        A = 10 ** (gain_db / 40)
        w0 = 2 * np.pi * f0 / sr
        alpha = np.sin(w0) / (2 * Q)
        b0 =  1 + alpha * A
        b1 = -2 * np.cos(w0)
        b2 =  1 - alpha * A
        a0 =  1 + alpha / A
        a1 = -2 * np.cos(w0)
        a2 =  1 - alpha / A
        return np.array([b0, b1, b2]) / a0, np.array([1, a1/a0, a2/a0])

    def process(self, signal: np.ndarray, sample_rate: int) -> np.ndarray:
        x = signal.astype(np.float32)
        sr = sample_rate
        # Low shelf
        b, a = self._low_shelf_coeffs(self.low_shelf_freq, self.low_shelf_gain_db, sr)
        x = sig.lfilter(b, a, x).astype(np.float32)
        # 3 peaking bands
        for f, g, q in [(self.peak1_freq, self.peak1_gain_db, self.peak1_q),
                        (self.peak2_freq, self.peak2_gain_db, self.peak2_q),
                        (self.peak3_freq, self.peak3_gain_db, self.peak3_q)]:
            b, a = self._peaking_coeffs(f, g, q, sr)
            x = sig.lfilter(b, a, x).astype(np.float32)
        # High shelf
        b, a = self._high_shelf_coeffs(self.high_shelf_freq, self.high_shelf_gain_db, sr)
        x = sig.lfilter(b, a, x).astype(np.float32)
        return np.clip(x, -1, 1).astype(np.float32)

dry = make_test_signal(1.0)
peq = ParametricEQ(low_shelf_gain_db=6, peak2_gain_db=-3, high_shelf_gain_db=3)
wet = peq.process(dry, SR)
print('ParametricEQ OK')
# Frequency response plot
freqs = np.linspace(20, 20000, 2048)
fig, ax = plt.subplots(figsize=(10, 3))
f_all = np.fft.rfftfreq(len(dry), 1/SR)
mag_dry = 20 * np.log10(np.abs(np.fft.rfft(dry)) + 1e-10)
mag_wet = 20 * np.log10(np.abs(np.fft.rfft(wet)) + 1e-10)
ax.semilogx(f_all[1:], mag_wet[1:] - mag_dry[1:], label='EQ curve', color='navy')
ax.set_xlabel('Frequency (Hz)'); ax.set_ylabel('Gain (dB)')
ax.set_title('ParametricEQ Frequency Response'); ax.grid(True, alpha=0.3)
ax.legend(); plt.tight_layout(); plt.show()

## Effect 12: GraphicEQ

A 10-band graphic EQ at ISO standard center frequencies (31, 63, 125, 250, 500, 1k, 2k, 4k, 8k, 16k Hz). Each band is a second-order peaking filter.

In [ ]:
class GraphicEQ(AudioEffect):
    """
    10-band ISO graphic equalizer.
    Center frequencies: 31, 63, 125, 250, 500, 1k, 2k, 4k, 8k, 16k Hz.
    Each band uses a second-order peaking biquad filter with fixed Q=1.41.
    gains_db: list of 10 gain values in dB, range -12 to +12.
    """
    ISO_FREQS = [31.25, 62.5, 125.0, 250.0, 500.0, 1000.0, 2000.0, 4000.0, 8000.0, 16000.0]

    def __init__(self, gains_db: list = None):
        super().__init__('graphic_eq')
        if gains_db is None:
            gains_db = [0.0] * 10
        self.gains_db = [np.clip(g, -12, 12) for g in gains_db[:10]]
        while len(self.gains_db) < 10:
            self.gains_db.append(0.0)

    def _peaking_biquad(self, f0, gain_db, Q, sr):
        A = 10 ** (gain_db / 40)
        w0 = 2 * np.pi * f0 / sr
        alpha = np.sin(w0) / (2 * Q)
        b0 = 1 + alpha*A; b1 = -2*np.cos(w0); b2 = 1 - alpha*A
        a0 = 1 + alpha/A; a1 = -2*np.cos(w0); a2 = 1 - alpha/A
        return [b0/a0, b1/a0, b2/a0], [1, a1/a0, a2/a0]

    def process(self, signal: np.ndarray, sample_rate: int) -> np.ndarray:
        x = signal.astype(np.float32)
        for f, g in zip(self.ISO_FREQS, self.gains_db):
            if f < sample_rate / 2 and abs(g) > 0.01:
                b, a = self._peaking_biquad(f, g, 1.41, sample_rate)
                x = sig.lfilter(b, a, x).astype(np.float32)
        return np.clip(x, -1, 1).astype(np.float32)

dry = make_test_signal(1.0)
geq = GraphicEQ(gains_db=[6, 4, 2, 0, -2, -4, -2, 0, 2, 4])
wet = geq.process(dry, SR)
print('GraphicEQ OK')
print(f'Params: {geq.get_params()}')

## Effect 13: Compressor

Dynamic range compressor using RMS or peak detection with programmable attack/release envelopes. The gain computer applies the compression curve, and the level detector provides smooth gain control.

In [ ]:
class Compressor(AudioEffect):
    """
    Dynamic range compressor.
    Uses RMS or peak level detection with attack/release smoothing.
    The soft knee smoothly transitions from uncompressed to compressed.
    makeup_gain compensates for gain reduction.
    """
    def __init__(self, threshold_db: float = -20.0, ratio: float = 4.0,
                 attack_ms: float = 10.0, release_ms: float = 100.0,
                 knee_db: float = 6.0, makeup_gain_db: float = 6.0,
                 detection: str = 'rms'):
        super().__init__('compressor')
        self.threshold_db = threshold_db
        self.ratio = max(1.0, ratio)
        self.attack_ms = attack_ms
        self.release_ms = release_ms
        self.knee_db = knee_db
        self.makeup_gain_db = makeup_gain_db
        self.detection = detection

    def process(self, signal: np.ndarray, sample_rate: int) -> np.ndarray:
        x = signal.astype(np.float64)
        attack_coef = np.exp(-1.0 / (self.attack_ms * 0.001 * sample_rate))
        release_coef = np.exp(-1.0 / (self.release_ms * 0.001 * sample_rate))
        T = self.threshold_db
        R = self.ratio
        W = self.knee_db
        makeup = 10 ** (self.makeup_gain_db / 20)
        # Level detection
        if self.detection == 'rms':
            window = max(1, int(0.01 * sample_rate))
            sq = x ** 2
            from scipy.ndimage import uniform_filter1d
            rms = np.sqrt(uniform_filter1d(sq, window) + 1e-10)
            level_db = 20 * np.log10(rms + 1e-10)
        else:
            level_db = 20 * np.log10(np.abs(x) + 1e-10)
        # Gain computer (soft knee)
        gain_db = np.zeros_like(level_db)
        for i in range(len(level_db)):
            lvl = level_db[i]
            if 2 * (lvl - T) < -W:
                gain_db[i] = 0.0
            elif 2 * abs(lvl - T) <= W:
                gain_db[i] = (1/R - 1) * (lvl - T + W/2)**2 / (2 * W)
            else:
                gain_db[i] = (1/R - 1) * (lvl - T)
        # Smooth gain with attack/release
        smooth_gain = np.zeros_like(gain_db)
        prev = 0.0
        for i in range(len(gain_db)):
            if gain_db[i] < prev:
                smooth_gain[i] = attack_coef * prev + (1 - attack_coef) * gain_db[i]
            else:
                smooth_gain[i] = release_coef * prev + (1 - release_coef) * gain_db[i]
            prev = smooth_gain[i]
        gain_linear = 10 ** (smooth_gain / 20) * makeup
        out = x * gain_linear
        return np.clip(out, -1, 1).astype(np.float32)

dry = make_test_signal(2.0)
comp = Compressor(threshold_db=-20, ratio=4.0, attack_ms=5, release_ms=50, makeup_gain_db=8)
wet = comp.process(dry, SR)
print('Compressor OK')
plot_before_after(dry, wet, SR, 'Compressor')

## Effect 14: NoiseGate

A noise gate only passes signal above a threshold, suppressing quiet passages (background noise, low-level bleed). The attack, hold, and release phases control how quickly the gate opens and closes.

In [ ]:
class NoiseGate(AudioEffect):
    """
    Noise gate with attack/hold/release envelope.
    Signal below threshold is attenuated by floor_db.
    Attack: how quickly gate opens when signal exceeds threshold.
    Hold: minimum time gate stays open after signal drops below threshold.
    Release: how quickly gate closes after hold time expires.
    """
    def __init__(self, threshold_db: float = -40.0, attack_ms: float = 1.0,
                 hold_ms: float = 50.0, release_ms: float = 100.0,
                 floor_db: float = -80.0):
        super().__init__('noise_gate')
        self.threshold_db = threshold_db
        self.attack_ms = attack_ms
        self.hold_ms = hold_ms
        self.release_ms = release_ms
        self.floor_db = floor_db

    def process(self, signal: np.ndarray, sample_rate: int) -> np.ndarray:
        x = signal.astype(np.float32)
        threshold_lin = 10 ** (self.threshold_db / 20)
        floor_lin = 10 ** (self.floor_db / 20)
        attack_samples = max(1, int(self.attack_ms * 0.001 * sample_rate))
        hold_samples = max(1, int(self.hold_ms * 0.001 * sample_rate))
        release_samples = max(1, int(self.release_ms * 0.001 * sample_rate))
        # States: 0=closed, 1=attack, 2=open, 3=hold, 4=release
        env = np.zeros(len(x))
        state = 0  # closed
        hold_count = 0
        gate_level = floor_lin
        attack_step = (1.0 - floor_lin) / attack_samples
        release_step = (1.0 - floor_lin) / release_samples
        for i in range(len(x)):
            level = abs(x[i])
            if level > threshold_lin:
                if state in [0, 4]:
                    state = 1  # attack
                elif state == 1:
                    gate_level = min(1.0, gate_level + attack_step)
                    if gate_level >= 1.0:
                        state = 2
                elif state == 2:
                    hold_count = 0
                state = 2 if state == 3 else state
                if state == 1:
                    gate_level = min(1.0, gate_level + attack_step)
                    if gate_level >= 1.0: state = 2
            else:
                if state == 2:
                    state = 3; hold_count = 0
                elif state == 3:
                    hold_count += 1
                    if hold_count >= hold_samples:
                        state = 4
                elif state == 4:
                    gate_level = max(floor_lin, gate_level - release_step)
                    if gate_level <= floor_lin: state = 0
            env[i] = gate_level
        return (x * env).astype(np.float32)

# Create signal with quiet sections
t = np.linspace(0, 2.0, int(2.0 * SR), endpoint=False)
test_sig = (0.8 * np.sin(2 * np.pi * 440 * t)).astype(np.float32)
test_sig[int(0.5*SR):int(0.8*SR)] *= 0.01  # quiet section
test_sig[int(1.2*SR):int(1.5*SR)] *= 0.01
ng = NoiseGate(threshold_db=-30, hold_ms=50, release_ms=100)
gated = ng.process(test_sig, SR)
print('NoiseGate OK')
plot_before_after(test_sig, gated, SR, 'NoiseGate', duration_show=2.0)

## Effect 15: Limiter

A lookahead limiter uses a circular buffer to look ahead in the audio stream, allowing smooth gain reduction before peaks are reached. This prevents clipping while avoiding the audible distortion of hard clipping.

In [ ]:
class Limiter(AudioEffect):
    """
    Lookahead limiter using a circular delay buffer.
    The lookahead allows gain reduction to begin before the peak arrives,
    preventing clipping while maintaining transparency.
    threshold_db: gain reduction starts when signal exceeds this level.
    lookahead_ms: how far ahead to look for peaks.
    """
    def __init__(self, threshold_db: float = -1.0, lookahead_ms: float = 5.0,
                 release_ms: float = 50.0):
        super().__init__('limiter')
        self.threshold_db = threshold_db
        self.lookahead_ms = lookahead_ms
        self.release_ms = release_ms

    def process(self, signal: np.ndarray, sample_rate: int) -> np.ndarray:
        x = signal.astype(np.float64)
        threshold = 10 ** (self.threshold_db / 20)
        lookahead = max(1, int(self.lookahead_ms * 0.001 * sample_rate))
        release_coef = np.exp(-1.0 / (self.release_ms * 0.001 * sample_rate))
        # Delay signal by lookahead
        delayed = np.concatenate([np.zeros(lookahead), x])
        # Compute gain reduction from future peaks
        gain = np.ones(len(x))
        peak_hold = 0.0
        for i in range(len(x)):
            # Look ahead window
            window_end = min(i + lookahead, len(x))
            peak = np.max(np.abs(x[i:window_end]))
            if peak > threshold:
                peak_hold = max(peak_hold, threshold / (peak + 1e-10))
            else:
                peak_hold = min(1.0, peak_hold + (1.0 - peak_hold) * (1 - release_coef))
            gain[i] = peak_hold
        # Smooth gain
        gain_smooth = np.zeros_like(gain)
        prev = 1.0
        for i in range(len(gain)):
            if gain[i] < prev:
                gain_smooth[i] = gain[i]  # instant attack
            else:
                gain_smooth[i] = release_coef * prev + (1-release_coef) * gain[i]
            prev = gain_smooth[i]
        out = delayed[lookahead:lookahead+len(x)] * gain_smooth
        return np.clip(out, -1, 1).astype(np.float32)

t = np.linspace(0, 1.0, SR, endpoint=False)
test_hot = (1.5 * np.sin(2 * np.pi * 440 * t)).astype(np.float32)  # over-threshold
lim = Limiter(threshold_db=-1.0, lookahead_ms=5.0, release_ms=30.0)
limited = lim.process(test_hot, SR)
print(f'Limiter OK. Input peak: {np.max(np.abs(test_hot)):.3f}, Output peak: {np.max(np.abs(limited)):.3f}')
plot_before_after(test_hot, limited, SR, 'Limiter')

In [ ]:
# Summary of all effect classes
all_effects = [
    ConvolutionReverb, SchroederReverb, MoorerReverb, DigitalDelay,
    Chorus, Flanger, Phaser, Tremolo, Distortion, BitCrusher,
    ParametricEQ, GraphicEQ, Compressor, NoiseGate, Limiter
]
print('=' * 50)
print('NOTEBOOK 01 COMPLETE')
print('=' * 50)
print(f'Total effects implemented: {len(all_effects)}')
print()
for cls in all_effects:
    e = cls()
    print(f'  {cls.__name__:<20} params: {list(e.get_params().keys())[:5]}')
print('\nCheckmark: Notebook complete')